# Solutions — Generators & Iterators: Lazy Streaming

Exercise notebook: `03_pythonic_python/05_generators_iterators.ipynb`. Each solution has: working code, an explanation, complexity, common mistakes, and a Java-style vs Pythonic comparison where it teaches something.

## Exercise 1 — 🟢 Easy · Read meaningful lines lazily

**Working solution**

In [ ]:
def read_lines(path):
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line and not line.startswith("#"):
                yield line

**Explanation**

Iterating a file object reads line by line (buffered), never the whole file. `with` inside a generator closes the file when the generator is exhausted or garbage-collected.

**Complexity:** O(1) memory.

**Common mistakes**

- `f.readlines()` — loads everything.
- Returning a list 'just in case' — kills the streaming benefit.

## Exercise 2 — 🟡 Medium · Batch any iterable

**Working solution**

In [ ]:
from itertools import islice

def batched(iterable, n):
    if n < 1:
        raise ValueError("n must be >= 1")
    iterator = iter(iterable)
    while batch := tuple(islice(iterator, n)):
        yield batch

**Explanation**

`islice` pulls at most `n` items from the *same* iterator each time; an empty tuple ends the loop. (Python 3.12 ships
`itertools.batched`; writing it yourself teaches how iterators are consumed.)

**Complexity:** O(n) memory per batch.

**Common mistakes**

- Calling `islice(iterable, n)` on a list repeatedly — a list restarts from the beginning each time; you must call `iter()` once.

## Exercise 3 — 🟡 Medium · Tolerant JSON Lines parser

**Working solution**

In [ ]:
import json

def iter_jsonl(lines, errors):
    for line_no, line in enumerate(lines, start=1):
        if not line.strip():
            continue
        try:
            obj = json.loads(line)
        except json.JSONDecodeError as exc:
            errors.append((line_no, exc.msg))
            continue
        if not isinstance(obj, dict):
            errors.append((line_no, f"expected object, got {type(obj).__name__}"))
            continue
        yield obj

**Explanation**

Generators can't easily 'return' side information while streaming, so an explicit error sink (a list, a logger, a dead-letter writer) is the common pattern.

**Complexity:** O(1) memory per line.

**Common mistakes**

- Catching `Exception` around `yield` — exceptions from the *consumer* don't come through there, but bugs in your own parsing would be hidden.

## Exercise 4 — 🟠 Hard · A class-based paginating iterator

**Working solution**

In [ ]:
class PageIterator:
    def __init__(self, fetch_page, start_page=1):
        self._fetch = fetch_page
        self._next_page = start_page
        self._buffer = []
        self._pos = 0
        self._done = False
        self.pages_fetched = 0

    def __iter__(self):
        return self

    def __next__(self):
        while self._pos >= len(self._buffer):
            if self._done:
                raise StopIteration
            self._buffer = self._fetch(self._next_page)
            self.pages_fetched += 1
            self._next_page += 1
            self._pos = 0
            if not self._buffer:
                self._done = True
                raise StopIteration
        item = self._buffer[self._pos]
        self._pos += 1
        return item

**Explanation**

This is the iterator protocol a `for` loop uses under the hood: `iter(obj)` then `next()` until `StopIteration`.
Writing it once by hand demystifies generators — which give you all of this for free. In real code you'd write
`def iter_pages(fetch): page = 1; while batch := fetch(page): yield from batch; page += 1`.

**Complexity:** O(page size) memory.

**Common mistakes**

- Fetching all pages in `__init__` (eager).
- Not handling empty pages in the middle — here an empty page means 'end'.

## Challenge — 🔴 Very Hard · Streaming event analytics on a large file

**Working solution**

In [ ]:
import json
from collections import Counter

def _parse(lines, errors):
    for line_no, line in enumerate(lines, start=1):
        if not line.strip():
            continue
        try:
            obj = json.loads(line)
        except json.JSONDecodeError:
            errors.append(line_no)
            continue
        yield obj if isinstance(obj, dict) else {}

def _counting(lines, stats):
    for line in lines:
        if line.strip():
            stats["lines"] += 1
            yield line

def event_stats(path):
    errors = []
    stats = {"lines": 0, "malformed": 0, "invalid": 0, "valid": 0}
    by_type, users = Counter(), set()
    revenue = 0.0
    first_ts = last_ts = None
    with open(path, encoding="utf-8") as f:
        for event in _parse(_counting(f, stats), errors):
            if not event.get("user_id") or not event.get("ts"):
                stats["invalid"] += 1
                continue
            stats["valid"] += 1
            by_type[event.get("event_type")] += 1
            users.add(event["user_id"])
            ts = event["ts"]
            first_ts = ts if first_ts is None or ts < first_ts else first_ts
            last_ts = ts if last_ts is None or ts > last_ts else last_ts
            if event.get("event_type") == "purchase":
                revenue += float((event.get("properties") or {}).get("amount") or 0)
    stats["malformed"] = len(errors)
    return {**stats, "by_type": dict(sorted(by_type.items(), key=lambda kv: (-kv[1], str(kv[0])))),
            "purchase_revenue": round(revenue, 2), "active_users": len(users), "first_ts": first_ts, "last_ts": last_ts}

**Explanation**

Each stage is a generator, so only one line is in flight at a time; aggregates (`Counter`, a set of ids, running min/max)
are the only state. `_counting` is a pass-through stage that observes the stream without buffering it — the generator
equivalent of a Spark accumulator. That's exactly how Spark executes a narrow transformation chain within a partition.

**Complexity:** O(n) time, O(distinct users + event types) memory.

**Common mistakes**

- `json.load(f)` / `f.read().splitlines()` → whole file in memory.
- Keeping all events in a list 'to compute stats later'.